In [8]:
from pathlib import Path
from PIL import Image
import shutil, os, json, subprocess

# Copy project code
CODE_SRC = Path("/kaggle/input/datasets/dmmehedihasanabid/icmla26-updated-code/icmla26_code")
CODE_DST = Path("/kaggle/working/icmla26_code")

if CODE_DST.exists():
    shutil.rmtree(CODE_DST)

shutil.copytree(CODE_SRC, CODE_DST)
os.chdir(CODE_DST)

subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
subprocess.run("python check_environment.py", shell=True, check=True)

# Prepare clean standardized dataset folders
PREP = Path("/kaggle/working/icmla_prepared")
if PREP.exists():
    shutil.rmtree(PREP)
PREP.mkdir(parents=True, exist_ok=True)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def safe_link_tree(src_dir, dst_dir, verify=False, prefix=""):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    bad = 0

    for i, p in enumerate(sorted(src_dir.glob("*"))):
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
            continue

        if verify:
            try:
                with Image.open(p) as img:
                    img.verify()
            except Exception:
                bad += 1
                continue

        safe_prefix = f"{prefix}_" if prefix else ""
        out = dst_dir / f"{safe_prefix}{i:06d}_{p.name}"

        os.symlink(p, out)
        copied += 1

    return copied, bad

# 1. PlantVillage source
PV_ROOT = Path("/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage")

print("PlantVillage")
print("Early", safe_link_tree(PV_ROOT / "Potato___Early_blight", PREP / "plantvillage_potato/Early Blight", prefix="pv_early"))
print("Late", safe_link_tree(PV_ROOT / "Potato___Late_blight", PREP / "plantvillage_potato/Late Blight", prefix="pv_late"))
print("Healthy", safe_link_tree(PV_ROOT / "Potato___healthy", PREP / "plantvillage_potato/Healthy", prefix="pv_healthy"))

# 2. PLD Pakistan target
PLD_ROOT = Path("/kaggle/input/datasets/rizwan123456789/potato-disease-leaf-datasetpld/PLD_3_Classes_256")

print("PLD Pakistan")
for split in ["Training", "Validation", "Testing"]:
    print(split, "Early", safe_link_tree(PLD_ROOT / split / "Early_Blight", PREP / "pld_pakistan/Early Blight", verify=True, prefix=f"pld_{split}_early"))
    print(split, "Late", safe_link_tree(PLD_ROOT / split / "Late_Blight", PREP / "pld_pakistan/Late Blight", verify=True, prefix=f"pld_{split}_late"))
    print(split, "Healthy", safe_link_tree(PLD_ROOT / split / "Healthy", PREP / "pld_pakistan/Healthy", verify=True, prefix=f"pld_{split}_healthy"))

# 3. Irish target
IRISH_ROOT = Path("/kaggle/input/datasets/dmmehedihasanabid/irish-potato/Irish for Kaggle")

print("Irish")
for src_cls, dst_cls, pref in [
    ("Early blight", "Early Blight", "irish_early"),
    ("Late blight", "Late Blight", "irish_late"),
    ("Healthy", "Healthy", "irish_healthy")
]:
    copied, bad = safe_link_tree(IRISH_ROOT / src_cls, PREP / f"irish_potato/{dst_cls}", verify=True, prefix=pref)
    print(src_cls, "copied:", copied, "bad:", bad)

# 4. PlantDoc potato-only overlap
PLANTDOC_ROOT = Path("/kaggle/input/datasets/abdulhasibuddin/plant-doc-dataset/PlantDoc-Dataset")

print("PlantDoc potato overlap")
for split in ["train", "test"]:
    print(split, "Early", safe_link_tree(PLANTDOC_ROOT / split / "Potato leaf early blight", PREP / "plantdoc_potato_overlap/Early Blight", verify=True, prefix=f"plantdoc_{split}_early"))
    print(split, "Late", safe_link_tree(PLANTDOC_ROOT / split / "Potato leaf late blight", PREP / "plantdoc_potato_overlap/Late Blight", verify=True, prefix=f"plantdoc_{split}_late"))

# Canonical config
cfg = {
    "inherits": "configs/default_config.json",
    "output_dir": "/kaggle/working/paper_outputs",
    "datasets": {
        "source": {
            "name": "PlantVillage Potato",
            "domain": "laboratory",
            "path": "/kaggle/working/icmla_prepared/plantvillage_potato"
        },
        "targets": [
            {
                "name": "PLD Pakistan",
                "domain": "regional_field_cropped",
                "path": "/kaggle/working/icmla_prepared/pld_pakistan"
            },
            {
                "name": "Irish Potato Dataset",
                "domain": "field",
                "path": "/kaggle/working/icmla_prepared/irish_potato",
                "fixed_split_per_class": {
                    "train": 998,
                    "val": 200,
                    "test": 300,
                    "seed": 2026
                }
            },
            {
                "name": "PlantDoc Potato Overlap",
                "domain": "uncontrolled_field_partial_overlap",
                "path": "/kaggle/working/icmla_prepared/plantdoc_potato_overlap",
                "optional": True
            }
        ]
    }
}

Path("configs/kaggle_config.json").write_text(json.dumps(cfg, indent=2))

print("Setup complete.")
print(Path("configs/kaggle_config.json").read_text())

# Final count check
print("\nFinal prepared counts:")
for d in sorted(PREP.rglob("*")):
    if d.is_dir() and d.parent != PREP and len(list(d.glob("*"))) > 0:
        print(len(list(d.glob("*"))), d)


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OK required numpy: 2.0.2
OK required pandas: 2.3.3
OK required scipy: 1.16.3
OK required sklearn: 1.6.1
OK required PIL: 11.3.0
OK required matplotlib: 3.10.0
OK required seaborn: 0.13.2
OK required torch: 2.10.0+cu128
OK required torchvision: 0.25.0+cu128
OK optional cv2: 4.13.0
OK optional timm: 1.0.26
OK optional open_clip: 3.3.0
OK optional transformers: 5.0.0
OK optional umap: 0.5.12
PlantVillage
Early (1000, 0)
Late (1000, 0)
Healthy (152, 0)
PLD Pakistan
Training Early (1303, 0)
Training Late (1132, 0)
Training Healthy (816, 0)
Validation Early (163, 0)
Validation Late (151, 0)
Validation Healthy (102, 0)
Testing Early (162, 0)
Testing Late (141, 0)
Testing Healthy (102, 0)
Irish
Early blight copied: 1498 bad: 2
Late blight copied: 1500 bad: 0
Healthy copied: 1500 bad: 0
PlantDoc potato overlap
train Early (109, 0)
train Late (97, 0)
test Early (8, 0)
test Late (8, 0)
Setup complete.
{
  "inherits": "configs/default_conf

In [9]:
%cd /kaggle/working/icmla26_code

/kaggle/working


In [10]:
!pwd
!ls src/pipeline.py

/kaggle/working/icmla26_code
src/pipeline.py


In [11]:
from pathlib import Path

p = Path("src/pipeline.py")
text = p.read_text()

old = '''train_torch_model(adapted, fs_loader, target_val_loader, device, max(1, epochs // 2), cfg["learning_rate"], cfg["weight_decay"], paths["models"] / f"{model_name}_{slug(ds_name)}_{k}shot_seed{seed}.pt")'''

new = '''tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_{k}shot_seed{seed}.pt"
train_torch_model(adapted, fs_loader, target_val_loader, device, max(1, epochs // 2), cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
if tmp_ckpt.exists():
    tmp_ckpt.unlink()'''

old2 = '''train_torch_model(adapted, full_loader, target_val_loader, device, cfg["fine_tune_epochs"], cfg["learning_rate"], cfg["weight_decay"], paths["models"] / f"{model_name}_{slug(ds_name)}_full_ft.pt")'''

new2 = '''tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_full_ft.pt"
train_torch_model(adapted, full_loader, target_val_loader, device, cfg["fine_tune_epochs"], cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
if tmp_ckpt.exists():
    tmp_ckpt.unlink()'''

if old not in text:
    print("WARNING: few-shot checkpoint line not found. It may already be patched.")
else:
    text = text.replace(old, new)
    print("Patched few-shot checkpoint saving.")

if old2 not in text:
    print("WARNING: full fine-tune checkpoint line not found. It may already be patched.")
else:
    text = text.replace(old2, new2)
    print("Patched full fine-tune checkpoint saving.")

p.write_text(text)

print("Done. ViT few-shot/full fine-tune checkpoints will be deleted after use.")

Patched few-shot checkpoint saving.
Patched full fine-tune checkpoint saving.
Done. ViT few-shot/full fine-tune checkpoints will be deleted after use.


In [12]:
!grep -n "tmp_ckpt" src/pipeline.py | head -20

1274:                    tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_{k}shot_seed{seed}.pt"
1275:train_torch_model(adapted, fs_loader, target_val_loader, device, max(1, epochs // 2), cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
1276:if tmp_ckpt.exists():
1277:    tmp_ckpt.unlink()
1292:                tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_full_ft.pt"
1293:train_torch_model(adapted, full_loader, target_val_loader, device, cfg["fine_tune_epochs"], cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
1294:if tmp_ckpt.exists():
1295:    tmp_ckpt.unlink()


In [13]:
!rm -rf /kaggle/working/paper_outputs_stage2_vit
!du -sh /kaggle/working

44M	/kaggle/working


In [14]:
from pathlib import Path
import shutil, os

%cd /kaggle/working/icmla26_code

SRC = Path("/kaggle/input/datasets/dmmehedihasanabid/icmla26-updated-code/icmla26_code/src/pipeline.py")
DST = Path("src/pipeline.py")

shutil.copy2(SRC, DST)
print("Restored:", DST)

/kaggle/working
Restored: src/pipeline.py


In [15]:
from pathlib import Path

p = Path("src/pipeline.py")
text = p.read_text()

old = '''                    train_torch_model(adapted, fs_loader, target_val_loader, device, max(1, epochs // 2), cfg["learning_rate"], cfg["weight_decay"], paths["models"] / f"{model_name}_{slug(ds_name)}_{k}shot_seed{seed}.pt")'''

new = '''                    tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_{k}shot_seed{seed}.pt"
                    train_torch_model(adapted, fs_loader, target_val_loader, device, max(1, epochs // 2), cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
                    if tmp_ckpt.exists():
                        tmp_ckpt.unlink()'''

old2 = '''                train_torch_model(adapted, full_loader, target_val_loader, device, cfg["fine_tune_epochs"], cfg["learning_rate"], cfg["weight_decay"], paths["models"] / f"{model_name}_{slug(ds_name)}_full_ft.pt")'''

new2 = '''                tmp_ckpt = paths["models"] / f"_tmp_{model_name}_{slug(ds_name)}_full_ft.pt"
                train_torch_model(adapted, full_loader, target_val_loader, device, cfg["fine_tune_epochs"], cfg["learning_rate"], cfg["weight_decay"], tmp_ckpt)
                if tmp_ckpt.exists():
                    tmp_ckpt.unlink()'''

if old not in text:
    print("Few-shot line not found.")
else:
    text = text.replace(old, new)
    print("Patched few-shot checkpoints.")

if old2 not in text:
    print("Full fine-tune line not found.")
else:
    text = text.replace(old2, new2)
    print("Patched full fine-tune checkpoints.")

p.write_text(text)
print("Patch complete.")

Patched few-shot checkpoints.
Patched full fine-tune checkpoints.
Patch complete.


In [16]:
!python -m py_compile src/pipeline.py

In [17]:
!rm -rf /kaggle/working/paper_outputs_stage2_vit
!rm -f /kaggle/working/stage2_vit_outputs.zip
!du -sh /kaggle/working

44M	/kaggle/working


In [18]:
from pathlib import Path
import json, subprocess

STAGE_NAME = "stage2_vit"
MODELS = ["vit_b_16"]
BATCH_SIZE = 4
OUTPUT_DIR = f"/kaggle/working/paper_outputs_{STAGE_NAME}"

stage_cfg = {
    "inherits": "configs/kaggle_config.json",
    "output_dir": OUTPUT_DIR,
    "models": MODELS,
    "batch_size": BATCH_SIZE
}

Path(f"configs/{STAGE_NAME}_config.json").write_text(json.dumps(stage_cfg, indent=2))

subprocess.run(f"rm -rf {OUTPUT_DIR}", shell=True, check=False)

subprocess.run(
    f"python run_pipeline.py --config configs/{STAGE_NAME}_config.json --skip-foundation",
    shell=True,
    check=True
)

subprocess.run(
    f'zip -r /kaggle/working/{STAGE_NAME}_outputs.zip "{OUTPUT_DIR}" > /dev/null',
    shell=True,
    check=True
)

print(f"Download: /kaggle/working/{STAGE_NAME}_outputs.zip")

Using device: cuda
Class mapping:
  0: Early Blight
  1: Late Blight
  2: Healthy

PlantVillage Potato: 2152 images
  Early Blight: 1000
  Late Blight: 1000
  Healthy: 152

PLD Pakistan: 4072 images
  Early Blight: 1628
  Late Blight: 1424
  Healthy: 1020

Irish Potato Dataset: 4498 images
  Early Blight: 1498
  Late Blight: 1500
  Healthy: 1500

PlantDoc Potato Overlap: 222 images
  Early Blight: 117
  Late Blight: 105
  Healthy: 0

Training vit_b_16
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 203MB/s]  
evidence vit_b_16: 100%|██████████| 64/64 [01:12<00:00,  1.14s/it]
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/kaggle/working/icmla26_code/src/pipeline.py:612: RuntimeWarning: invalid value encountered in cast
  arr[mask] = np.median(arr[~mask], axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/kaggle/working/icmla26_code/src/pipeline.py:612: RuntimeWarning: invalid value encountered in cast
  arr[mask] = np.median(a


Done. Outputs saved to /kaggle/working/paper_outputs_stage2_vit
Download: /kaggle/working/stage2_vit_outputs.zip


In [19]:
from IPython.display import FileLink, display
from pathlib import Path

zip_path = Path("/kaggle/working/stage2_vit_outputs.zip")

if zip_path.exists():
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    print(f"ZIP ready: {zip_path}")
    print(f"Size: {size_mb:.2f} MB")
    
    display(
        FileLink(
            str(zip_path),
            result_html_prefix="Click here to download: "
        )
    )
else:
    print("ZIP not found. Wait for the experiment and ZIP creation to finish.")

ZIP ready: /kaggle/working/stage2_vit_outputs.zip
Size: 367.95 MB


/kaggle/working/stage2_vit_outputs.zip

In [20]:
from pathlib import Path
from IPython.display import display, FileLink, Markdown, HTML
import shutil, os, glob, textwrap

# Change this if your stage name is different
STAGE_NAME = "stage2_vit"   # examples: stage1, stage2_resnet, stage2_vit, stage3_foundation

OUTPUT_DIR = Path(f"/kaggle/working/paper_outputs_{STAGE_NAME}")
ZIP_PATH = Path(f"/kaggle/working/{STAGE_NAME}_outputs.zip")

print("=" * 80)
print("OUTPUT CHECK")
print("=" * 80)

if not OUTPUT_DIR.exists():
    print(f"ERROR: Output folder not found: {OUTPUT_DIR}")
else:
    print(f"Found output folder: {OUTPUT_DIR}")

    # Recreate ZIP safely
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    print("Creating ZIP archive...")
    shutil.make_archive(
        str(ZIP_PATH).replace(".zip", ""),
        "zip",
        root_dir=OUTPUT_DIR.parent,
        base_dir=OUTPUT_DIR.name,
    )

    size_gb = ZIP_PATH.stat().st_size / (1024 ** 3)
    size_mb = ZIP_PATH.stat().st_size / (1024 ** 2)

    print(f"ZIP ready: {ZIP_PATH}")
    print(f"ZIP size: {size_gb:.2f} GB ({size_mb:.1f} MB)")

    display(Markdown("## Download Output ZIP"))
    display(FileLink(str(ZIP_PATH)))

    display(Markdown("## Important Output Files"))

    important_files = [
        OUTPUT_DIR / "paper_summary.txt",
        OUTPUT_DIR / "all_results.csv",
        OUTPUT_DIR / "seed_results.csv",
        OUTPUT_DIR / "config.json",
    ]

    for f in important_files:
        if f.exists:
            display(FileLink(str(f)))

    summary_file = OUTPUT_DIR / "paper_summary.txt"
    if summary_file.exists():
        display(Markdown("## Paper Summary"))
        txt = summary_file.read_text(errors="ignore")
        display(Markdown("```text\n" + txt[:12000] + "\n```"))

    display(Markdown("## Tables"))
    table_files = sorted((OUTPUT_DIR / "tables").glob("*")) if (OUTPUT_DIR / "tables").exists() else []
    for f in table_files[:30]:
        display(FileLink(str(f)))

    display(Markdown("## Figures"))
    fig_files = sorted(list((OUTPUT_DIR / "figures").glob("*.png"))) if (OUTPUT_DIR / "figures").exists() else []
    for f in fig_files[:20]:
        display(Markdown(f"### {f.name}"))
        display(HTML(f'<img src="{f}" style="max-width:900px; width:100%; border:1px solid #ddd;">'))

    display(Markdown("## Folder Size"))
    !du -sh "{OUTPUT_DIR}"
    !find "{OUTPUT_DIR}" -maxdepth 2 -type f | wc -l

print("=" * 80)
print("If you save the Kaggle notebook version now, this cell will preserve visible links/results.")
print("=" * 80)


OUTPUT CHECK
Found output folder: /kaggle/working/paper_outputs_stage2_vit
Creating ZIP archive...
ZIP ready: /kaggle/working/stage2_vit_outputs.zip
ZIP size: 0.36 GB (368.1 MB)


## Download Output ZIP

/kaggle/working/stage2_vit_outputs.zip

## Important Output Files

/kaggle/working/paper_outputs_stage2_vit/paper_summary.txt

...

### leaf_only_vs_full_image_vs_background_only_results.png

### lesion_focus_ratio.png

### lesion_occlusion_results.png

## Folder Size

398M	/kaggle/working/paper_outputs_stage2_vit
84
If you save the Kaggle notebook version now, this cell will preserve visible links/results.


In [ ]:
import os
import glob
import time

# Automatically find ZIP files created in /kaggle/working
zip_files = glob.glob("/kaggle/working/**/*.zip", recursive=True)

if zip_files:
    # Pick the most recently created/modified ZIP
    zip_path = max(zip_files, key=os.path.getmtime)

    size_mb = os.path.getsize(zip_path) / (1024 ** 2)

    print("\n" + "=" * 60)
    print("✅ ZIP IS READY!")
    print("📦", os.path.basename(zip_path))
    print(f"📏 Size: {size_mb:.2f} MB")
    print("=" * 60)

    # Alert every 5 minutes
    while True:
        print("\a\a\a")
        print("🔔🔔🔔 ZIP IS READY — DOWNLOAD IT NOW! 🔔🔔🔔")
        time.sleep(300)

else:
    print("❌ No ZIP file found in /kaggle/working/")


✅ ZIP IS READY!
📦 stage2_vit_outputs.zip
📏 Size: 368.09 MB

🔔🔔🔔 ZIP IS READY — DOWNLOAD IT NOW! 🔔🔔🔔


In [1]:
from pathlib import Path
from IPython.display import display, Markdown, HTML, FileLink
import pandas as pd
import os, glob, textwrap, time

STAGE_NAME = "stage2_vit"
OUT = Path(f"/kaggle/working/paper_outputs_{STAGE_NAME}")

display(Markdown(f"# {STAGE_NAME} Stage Report"))

if not OUT.exists():
    display(Markdown(f"**Output folder not found yet:** `{OUT}`"))
else:
    display(Markdown(f"**Output folder:** `{OUT}`"))

    print("\nFolder size:")
    os.system(f'du -sh "{OUT}"')

    print("\nTop-level files:")
    os.system(f'find "{OUT}" -maxdepth 1 -type f -print')

    summary = OUT / "paper_summary.txt"
    if summary.exists():
        display(Markdown("## Paper Summary"))
        txt = summary.read_text(errors="ignore")
        display(Markdown("```text\n" + txt[:15000] + "\n```"))

    all_results = OUT / "all_results.csv"
    if all_results.exists():
        display(Markdown("## All Results Preview"))
        df = pd.read_csv(all_results)
        display(df.head(50))
        display(Markdown("### Result Counts"))
        display(df.groupby(["model", "dataset", "experiment"]).size().reset_index(name="rows"))

    seed_results = OUT / "seed_results.csv"
    if seed_results.exists():
        display(Markdown("## Seed Results Preview"))
        seed_df = pd.read_csv(seed_results)
        display(seed_df.head(50))

    tables_dir = OUT / "tables"
    if tables_dir.exists():
        display(Markdown("## Main Tables"))
        table_names = [
            "dataset_summary.csv",
            "main_performance_table.csv",
            "few_shot_adaptation_table.csv",
            "attribution_evidence_table.csv",
            "lesion_occlusion_table.csv",
            "background_perturbation_table.csv",
            "leaf_only_background_only_table.csv",
            "representation_geometry_table.csv",
            "efficiency_table.csv",
            "statistical_significance_table.csv",
            "class_wise_performance_table.csv",
        ]

        for name in table_names:
            f = tables_dir / name
            if f.exists():
                display(Markdown(f"### {name}"))
                try:
                    d = pd.read_csv(f)
                    display(d.head(30))
                except Exception as e:
                    print(f"Could not display {f}: {e}")

    figs_dir = OUT / "figures"
    if figs_dir.exists():
        display(Markdown("## Figures"))
        figs = sorted(figs_dir.glob("*.png"))
        print(f"Found {len(figs)} PNG figures.")

        for f in figs[:25]:
            display(Markdown(f"### {f.name}"))
            display(HTML(f'<img src="{f}" style="max-width:950px; width:100%; border:1px solid #ccc; margin-bottom:20px;">'))

    display(Markdown("## Download Links For Individual Critical Files"))
    for f in [
        OUT / "paper_summary.txt",
        OUT / "all_results.csv",
        OUT / "seed_results.csv",
        OUT / "config.json",
    ]:
        if f.exists():
            display(FileLink(str(f)))

    if tables_dir.exists():
        for f in sorted(tables_dir.glob("*.csv")):
            display(FileLink(str(f)))

# stage2_vit Stage Report

**Output folder not found yet:** `/kaggle/working/paper_outputs_stage2_vit`